In [0]:
# Databricks notebook source
import json, os, time, urllib.parse, urllib.request
from datetime import date

API_KEY   = dbutils.secrets.get("stock-api", "alphavantage-key")
SYMBOLS   = ["AAPL", "MSFT", "NVDA", "AMZN"]
BASE_URL  = "https://www.alphavantage.co/query"
VOL       = "/Volumes/jarvis_demo_7405604983246276/stock/raw"
RUN_DATE  = date.today().isoformat()
THROTTLE  = 13          # 5 req/min limit
MAX_RETRY = 3

# COMMAND ----------

def fetch(params):
    """Fetch one endpoint. Returns dict, or None if the API refuses us."""
    url = f"{BASE_URL}?{urllib.parse.urlencode({**params, 'apikey': API_KEY})}"
    for attempt in range(1, MAX_RETRY + 1):
        with urllib.request.urlopen(url, timeout=30) as r:
            payload = json.load(r)

        # Alpha Vantage returns HTTP 200 for every failure mode. Inspect the body.
        if not payload:
            print(f"  empty body (attempt {attempt}) — backing off")
            time.sleep(20); continue
        if "Error Message" in payload:
            raise ValueError(f"Bad request for {params}: {payload['Error Message']}")
        if "Note" in payload or "Information" in payload:
            msg = payload.get("Note") or payload.get("Information")
            print(f"  throttled/limited (attempt {attempt}): {msg[:120]}")
            time.sleep(65); continue
        return payload

    print(f"  GIVING UP on {params}")
    return None


def clean_key(k):
    """'1. open' -> 'open'; 'Global Quote' -> 'global_quote'."""
    k = k.split(". ", 1)[-1] if k[0].isdigit() else k
    return k.strip().lower().replace(" ", "_").replace("%", "pct").replace("-", "_")


def write_lines(endpoint, symbol, rows, raw_payload):
    for sub, name, body in [
        ("_archive", f"{endpoint}", json.dumps(raw_payload, indent=2)),
        ("landing",  f"{endpoint}", "\n".join(json.dumps(r) for r in rows)),
    ]:
        d = f"{VOL}/{sub}/{name}/{RUN_DATE}" if sub == "_archive" else f"{VOL}/{sub}/{name}/{RUN_DATE}"
        os.makedirs(d, exist_ok=True)
        ext = "json" if sub == "_archive" else "jsonl"
        # overwrite is intentional: makes the daily run idempotent
        with open(f"{d}/{symbol}.{ext}", "w") as f:
            f.write(body)

# COMMAND ----------

def ingest_daily_prices(symbol):
    p = fetch({"function": "TIME_SERIES_DAILY", "symbol": symbol, "outputsize": "compact"})
    if not p or "Time Series (Daily)" not in p:
        return 0
    rows = [
        {"symbol": symbol, "trade_date": d, "run_date": RUN_DATE,
         **{clean_key(k): v for k, v in vals.items()}}
        for d, vals in p["Time Series (Daily)"].items()
    ]
    write_lines("daily_prices", symbol, rows, p)
    return len(rows)


def ingest_quote(symbol):
    p = fetch({"function": "GLOBAL_QUOTE", "symbol": symbol})
    q = (p or {}).get("Global Quote") or {}
    if not q:
        return 0
    row = {"run_date": RUN_DATE, **{clean_key(k): v for k, v in q.items()}}
    write_lines("quote", symbol, [row], p)
    return 1


def ingest_company(symbol):
    p = fetch({"function": "OVERVIEW", "symbol": symbol})
    if not p or "Symbol" not in p:
        return 0
    row = {"run_date": RUN_DATE, **{clean_key(k): v for k, v in p.items()}}
    write_lines("company", symbol, [row], p)
    return 1

# COMMAND ----------

summary, failures = [], []
for fn in (ingest_daily_prices, ingest_quote, ingest_company):
    for sym in SYMBOLS:
        print(f"{fn.__name__} :: {sym}")
        n = fn(sym)
        summary.append((fn.__name__, sym, n))
        if n == 0:
            failures.append((fn.__name__, sym))
        time.sleep(THROTTLE)

for name, sym, n in summary:
    print(f"{name:24} {sym:6} {n:>5} rows")

# Fail the task loudly so the job retries, but only if we got nothing useful at all.
if len(failures) == len(summary):
    raise RuntimeError(f"All ingestion calls failed: {failures}")
if failures:
    print(f"PARTIAL SUCCESS — failed: {failures}")
dbutils.notebook.exit(json.dumps({"run_date": RUN_DATE, "failures": failures}))

In [0]:
display(dbutils.fs.ls("/Volumes/jarvis_demo_7405604983246276/stock/raw/landing/daily_prices"))
print(spark.read.json("/Volumes/jarvis_demo_7405604983246276/stock/raw/landing/daily_prices/*/*.jsonl").count())  # ~400
spark.read.json("/Volumes/jarvis_demo_7405604983246276/stock/raw/landing/quote/*/*.jsonl").show(vertical=True)

path,name,size,modificationTime
dbfs:/Volumes/jarvis_demo_7405604983246276/stock/raw/landing/daily_prices/2026-08-18/,2026-08-18/,0,1787081066000


400
-RECORD 0------------------------
 change             | -15.0500   
 change_percent     | -3.0379%   
 high               | 492.6600   
 latest_trading_day | 2026-08-17 
 low                | 478.4100   
 open               | 490.1400   
 previous_close     | 495.4000   
 price              | 480.3500   
 run_date           | 2026-08-18 
 symbol             | MSFT       
 volume             | 29635848   
-RECORD 1------------------------
 change             | -0.3400    
 change_percent     | -0.1111%   
 high               | 307.6600   
 latest_trading_day | 2026-08-17 
 low                | 302.9390   
 open               | 306.2100   
 previous_close     | 305.9300   
 price              | 305.5900   
 run_date           | 2026-08-18 
 symbol             | AAPL       
 volume             | 38169263   
-RECORD 2------------------------
 change             | -1.3400    
 change_percent     | -0.5102%   
 high               | 265.7500   
 latest_trading_day | 2026-08-17 
 low      